# CardingForums — 04. Síntesis e informe

Este es el notebook final del caso CardingForums. Su propósito es **responder las preguntas de investigación** planteadas en el notebook 00, con evidencia generada en los notebooks 01-03.

Un buen informe forense:
1. **Separa lo que sabe con certeza** de lo que es hipótesis
2. **Cita la evidencia** en lugar de afirmar sin fundamento
3. **Reconoce las limitaciones** del método y los datos
4. **Es reproducible**: cualquiera que ejecute los notebooks anteriores llega a los mismos resultados

---

## 1. Setup y carga de resultados

Cargamos todos los outputs de los notebooks anteriores. Este notebook no genera datos nuevos — solo los consolida y presenta.

In [ ]:
import sys
sys.path.insert(0, "../")

import pandas as pd
import numpy as np
import json
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

from src.utils import RESULTS_DIR

# --- Cargar todos los resultados ---

# Notebook 00 — resumen del reconocimiento
recon_path = RESULTS_DIR / "00_reconocimiento_summary.json"
recon = json.loads(recon_path.read_text()) if recon_path.exists() else {}

# Notebook 01 — datos limpios
users = pd.read_parquet(RESULTS_DIR / "01_users_clean.parquet")
posts = pd.read_parquet(RESULTS_DIR / "01_posts_clean.parquet")

# Notebook 02 — análisis estructural
centrality = pd.read_parquet(RESULTS_DIR / "02_centrality.parquet")

# Notebook 03 — análisis semántico
profiles_path = RESULTS_DIR / "03_actor_profiles.parquet"
profiles = pd.read_parquet(profiles_path) if profiles_path.exists() else pd.DataFrame()

print("Resultados cargados:")
print(f"  Reconocimiento: {recon}")
print(f"  Usuarios: {len(users):,}")
print(f"  Posts: {len(posts):,}")
print(f"  Nodos con centralidad: {len(centrality):,}")
print(f"  Perfiles de actores: {len(profiles):,}")

## 2. Recapitulación de preguntas de investigación

Estas son las cinco preguntas que planteamos en el notebook 00. Vamos a responderlas una por una con la evidencia disponible.

---

## Pregunta 1: ¿Quiénes son los actores más activos?

**Metodología**: combinamos el conteo de posts de la tabla `user`, la centralidad de grado (cuántos usuarios diferentes frecuentan los mismos threads) y la betweenness centrality (quiénes actúan como puentes en la red).

**Hallazgo**: los usuarios con más posts no necesariamente son los más centrales en la red. Un vendedor con muchos posts en un solo subforo tiene bajo degree (interactúa con poca gente diversa) pero alta presencia volumétrica. Un administrador puede tener pocos posts pero muy alta betweenness (conecta muchos grupos).

**Limitación**: el conteo de posts de la tabla `user` puede diferir del conteo real si hay posts eliminados. El contador del campo `posts` lo actualiza el foro automáticamente y no siempre coincide con las filas reales en la tabla `post`.

In [ ]:
# Top 20 actores por volumen de posts
top_by_posts = (
    users[["forum", "username", "posts"]]
    .assign(posts=lambda df: pd.to_numeric(df["posts"], errors="coerce"))
    .sort_values("posts", ascending=False)
    .head(20)
    .reset_index(drop=True)
)

print("TOP 20 ACTORES POR VOLUMEN DE POSTS")
print(top_by_posts.to_string(index=False))

In [ ]:
# Top 20 por betweenness centrality (rol de puente)
if len(centrality):
    top_by_betweenness = (
        centrality[["username", "forum", "degree_centrality", "betweenness_centrality", "posts"]]
        .sort_values("betweenness_centrality", ascending=False)
        .head(20)
        .reset_index(drop=True)
    )
    print("TOP 20 ACTORES POR BETWEENNESS CENTRALITY (POSIBLES PUENTES/INTERMEDIARIOS)")
    print(top_by_betweenness.to_string(index=False))

In [ ]:
# Visualización: ranking combinado
if len(centrality):
    combined = centrality.nlargest(50, "betweenness_centrality")[
        ["username", "forum", "posts", "degree_centrality", "betweenness_centrality"]
    ].dropna(subset=["username"])

    fig = px.bar(
        combined.nlargest(20, "betweenness_centrality"),
        x="username", y="betweenness_centrality",
        color="forum",
        title="Top 20 actores por betweenness centrality",
        labels={"betweenness_centrality": "Betweenness centrality", "username": "Usuario"}
    )
    fig.update_xaxes(tickangle=45)
    fig.write_html(RESULTS_DIR / "04_top_betweenness.html")
    fig.show()

---

## Pregunta 2: ¿Hay roles diferenciados — vendedores, compradores, intermediarios?

**Metodología**: perfilado de actores con `qwen2.5:14b` (notebook 03), combinando NER, tópicos BERTopic y métricas de red.

**Hallazgo**: los roles más detectables son los vendedores (vocabulario muy específico: dumps, CVVs, precios, BINs) y los administradores/moderadores (tono diferente, referencias a reglas del foro, resolución de disputas). Los compradores son los más difíciles de identificar porque su texto es más genérico.

**Limitación**: la inferencia de roles es probabilística. Un mismo usuario puede actuar como vendedor en un foro y como comprador en otro. El nivel de confianza del modelo es una señal, no una certeza.

In [ ]:
if len(profiles):
    # Tabla de actores con rol y confianza
    print("TABLA DE ACTORES — ROLES INFERIDOS")
    print("=" * 80)
    
    cols = ["username", "forum", "role", "confidence", "summary"]
    available = [c for c in cols if c in profiles.columns]
    
    profiles_display = profiles[available].sort_values(
        ["confidence", "role"], ascending=[False, True]
    ).reset_index(drop=True)
    
    profiles_display
else:
    print("No hay perfiles disponibles. Ejecuta primero el notebook 03.")

In [ ]:
if len(profiles) and "role" in profiles.columns:
    # Distribución de roles con confianza media
    role_summary = profiles.groupby("role").agg(
        n_actores=("role", "count"),
        confianza_media=("confidence", "mean")
    ).reset_index().sort_values("n_actores", ascending=False)
    
    print("DISTRIBUCIÓN DE ROLES:")
    print(role_summary.to_string(index=False))
    
    fig = px.bar(
        role_summary,
        x="role", y="n_actores",
        color="confianza_media",
        title="Distribución de roles inferidos (top actores)",
        labels={"role": "Rol inferido", "n_actores": "Actores", "confianza_media": "Confianza media"},
        color_continuous_scale="RdYlGn"
    )
    fig.write_html(RESULTS_DIR / "04_role_distribution.html")
    fig.show()

---

## Pregunta 3: ¿Cómo se organiza la red de confianza?

**Metodología**: grafo de co-participación con NetworkX, detección de comunidades con Louvain, visualización con Plotly (notebook 02).

**Hallazgo**: la red tiene una componente gigante que agrupa a la mayoría de los usuarios activos, con un número reducido de comunidades bien definidas. Los nodos con alta betweenness conectan estas comunidades y son los candidatos más probables a actuar como intermediarios o personas de confianza transversal.

**Limitación**: la co-participación en threads es una proxy de interacción, no evidencia de comunicación directa. Dos usuarios en el mismo thread pueden no haber interactuado en absoluto.

In [ ]:
if len(centrality):
    # Resumen de la estructura de red
    print("ESTRUCTURA DE LA RED DE CO-PARTICIPACIÓN")
    print("=" * 50)
    print(f"  Usuarios en el análisis de red: {len(centrality):,}")
    
    if "community" in centrality.columns:
        n_communities = centrality["community"].nunique()
        print(f"  Comunidades detectadas: {n_communities}")
        
        # Tamaño promedio de comunidad
        community_sizes = centrality["community"].value_counts()
        print(f"  Comunidad más grande: {community_sizes.max()} usuarios")
        print(f"  Comunidad mediana: {community_sizes.median():.0f} usuarios")
    
    print(f"\n  Actor con mayor betweenness:")
    top_bridge = centrality.nlargest(1, "betweenness_centrality").iloc[0]
    print(f"    {top_bridge.get('username', 'N/A')} ({top_bridge.get('forum', '')})")
    print(f"    Betweenness: {top_bridge['betweenness_centrality']:.4f}")
    print(f"    Degree: {top_bridge['degree_centrality']:.4f}")

---

## Pregunta 4: ¿Hay usuarios que aparecen en múltiples foros?

**Metodología**: correlación por email, ICQ, username normalizado (notebook 01). Similitud de embeddings entre usuarios de distintos foros (notebook 03).

**Hallazgo**: la correlación por username (normalizado, sin distinguir mayúsculas) es la señal más abundante pero menos confiable — nombres comunes colisionan. La correlación por ICQ es más rara pero de muy alta confiabilidad. La similitud de embeddings permite detectar posibles identidades cruzadas incluso cuando el usuario cambia su nombre.

**Limitación**: la similitud de embeddings es una señal probabilística. Puede haber falsos positivos (dos personas con el mismo estilo de escritura) y falsos negativos (la misma persona que deliberadamente varía su estilo).

In [ ]:
# Correlación por ICQ (la señal más confiable)
if "icq" in users.columns:
    icq_matches = (
        users[users["icq"].notna() & (users["icq"] != "")]
        .groupby("icq")["forum"]
        .nunique()
        .pipe(lambda s: s[s > 1])
    )
    
    print(f"Usuarios correlacionados por ICQ en múltiples foros: {len(icq_matches):,} valores únicos")
    
    if len(icq_matches) > 0:
        # Detalle de los primeros matches
        icq_detail = (
            users[users["icq"].isin(icq_matches.index)]
            [["forum", "username", "icq", "posts"]]
            .sort_values(["icq", "posts"], ascending=[True, False])
            .head(30)
        )
        print("\nEjemplos de matches por ICQ:")
        print(icq_detail.to_string(index=False))
    else:
        print("No hay matches por ICQ en este dataset.")

# Correlación por username normalizado
if "username_normalized" in users.columns:
    username_matches = (
        users[users["username_normalized"].notna() & (users["username_normalized"].str.len() >= 4)]
        .groupby("username_normalized")["forum"]
        .nunique()
        .pipe(lambda s: s[s > 1])
    )
    print(f"\nUsernames únicos que aparecen en >1 foro: {len(username_matches):,}")

---

## Pregunta 5: ¿Qué se vende en cada subforo?

**Metodología**: TF-IDF sobre el texto de posts agrupado por subforo (notebook 02). NER sobre el corpus de usuarios activos (notebook 03).

**Hallazgo**: el TF-IDF revela vocabulario muy específico por subforo — términos como "track2", "dumps", "BIN" dominan en subforos de ventas, mientras que términos como "vouch", "escrow", "trusted" dominan en subforos de reputación. Esto valida la hipótesis de especialización temática.

**Limitación**: el TF-IDF no distingue contexto ("dumps" puede aparecer en una oferta o en una denuncia de estafa). El análisis semántico profundo de intención requiere NER y análisis de sentimiento adicional.

In [ ]:
# Mostrar entidades más frecuentes del NER (si están disponibles)
ner_path = RESULTS_DIR / "03_ner_results.jsonl"

if ner_path.exists():
    from collections import Counter
    
    ner_results = [json.loads(line) for line in ner_path.read_text().splitlines() if line.strip()]
    
    all_entities = []
    for r in ner_results:
        for e in r.get("entities", []):
            all_entities.append((e.get("type", "UNK"), e.get("text", "").lower()))
    
    entity_counts = Counter(all_entities)
    
    # Tabla por tipo de entidad
    print("ENTIDADES MÁS FRECUENTES POR TIPO (del NER)")
    print("=" * 60)
    
    for entity_type in ["PRODUCT", "PAYMENT", "TOOL", "MARKET", "COUNTRY", "CARD_TYPE"]:
        type_entities = [
            (text, count) for (etype, text), count in entity_counts.most_common(500)
            if etype == entity_type
        ][:10]
        if type_entities:
            print(f"\n{entity_type}:")
            for text, count in type_entities:
                print(f"  {text}: {count}")
else:
    print("Resultados de NER no disponibles. Ejecuta primero el notebook 03.")

## 3. Tabla consolidada de actores clave

Consolidamos toda la información en una tabla que resume lo que sabemos sobre cada actor relevante. Esta es la tabla principal del informe forense.

In [ ]:
if len(profiles) and len(centrality):
    # Unir perfiles con centralidad
    final_table = profiles.merge(
        centrality[["userid", "degree_centrality", "betweenness_centrality", "community"]],
        on="userid", how="left"
    )
    
    # Columnas para el informe final
    report_cols = [
        "username", "forum", "role", "confidence",
        "betweenness_centrality", "degree_centrality",
        "community", "summary", "signals", "caveats"
    ]
    available_report = [c for c in report_cols if c in final_table.columns]
    
    final_table_sorted = final_table[available_report].sort_values(
        "confidence", ascending=False
    )
    
    # Guardar tabla final
    final_table_sorted.to_parquet(RESULTS_DIR / "04_final_actor_table.parquet", index=False)
    final_table_sorted.to_csv(RESULTS_DIR / "04_final_actor_table.csv", index=False)
    
    print("TABLA FINAL DE ACTORES CLAVE")
    print("=" * 80)
    final_table_sorted
else:
    print("No hay datos suficientes para la tabla consolidada.")
    print("Verifica que los notebooks 02 y 03 se ejecutaron correctamente.")

## 4. Conclusiones

### Lo que sabemos con confianza

1. **La red tiene estructura de comunidades significativa** (modularidad > 0.3). No es ruido — hay grupos reales de usuarios que co-participan más entre sí que con el resto.

2. **Hay especialización temática por subforo**, visible en el TF-IDF: el vocabulario discrimina entre subforos de ventas, subforos de técnicas y subforos de reputación.

3. **Un pequeño número de usuarios concentra la mayoría de la actividad**. La distribución de posts sigue una ley de potencias: pocos usuarios con muchos posts, muchos usuarios con pocos posts.

4. **Los nodos con alta betweenness son candidatos a intermediarios de confianza**. Su posición estructural en la red es consistente con ese rol.

### Lo que queda como hipótesis

1. **La inferencia de roles por LLM es probable pero no verificable** sin datos externos (órdenes de compra, historial de transacciones, confirmaciones de terceros).

2. **La similitud de embeddings puede indicar identidades cruzadas**, pero necesita corroboración con señales adicionales (ICQ, email, IP).

3. **Los patrones horarios sugieren predominio geográfico**, pero no es concluyente sin validación contra los IPs registrados.

### Próximos pasos sugeridos

- Correlacionar los actores de alta betweenness con sus IPs registradas
- Analizar los mensajes privados (PMs) de los actores clave — probablemente contienen información más directa sobre transacciones
- Comparar los perfiles de este dataset con los de otros casos del curso

In [ ]:
# Resumen ejecutivo final
print("=" * 70)
print("RESUMEN EJECUTIVO — CARDINGFORUMS")
print("=" * 70)
print(f"  Foros analizados:    {recon.get('total_forums', 'N/A')}")
print(f"  Usuarios totales:    {recon.get('total_users', len(users)):,}")
print(f"  Posts totales:       {recon.get('total_posts', len(posts)):,}")
print(f"  Período cubierto:    2009 – 2021")

if len(centrality):
    top_actor = centrality.nlargest(1, "betweenness_centrality").iloc[0]
    print(f"\n  Actor central:       {top_actor.get('username', 'N/A')} ({top_actor.get('forum', '')})")

if len(profiles) and "role" in profiles.columns:
    role_counts = profiles["role"].value_counts()
    print(f"\n  Roles detectados:")
    for role, count in role_counts.items():
        print(f"    {role}: {count} actores")

print("\n  Archivos de resultados:")
for f in sorted(RESULTS_DIR.glob("*.parquet")) + sorted(RESULTS_DIR.glob("*.html")):
    print(f"    {f.name}")

## 5. Exportar el informe

### Opción A: HTML desde Jupyter

La forma más simple de exportar un notebook a un documento compartible:

```bash
# Desde la terminal, en la carpeta bloque4_cardingforums/
jupyter nbconvert --to html 04_sintesis_informe.ipynb
```

Esto genera `04_sintesis_informe.html` — un archivo autocontenido que incluye todos los gráficos estáticos y se puede abrir en cualquier navegador sin necesidad de Python.

**Limitación**: los gráficos de Plotly interactivos sí se incluyen (son JavaScript embebido). Los gráficos de Matplotlib se incluyen como imágenes.

### Opción B: PDF con nbconvert + LaTeX

```bash
# Requiere LaTeX instalado (texlive, MiKTeX, etc.)
jupyter nbconvert --to pdf 04_sintesis_informe.ipynb
```

El PDF es mejor para compartir con personas externas que no tienen Jupyter. La desventaja es que los gráficos interactivos de Plotly no funcionan en PDF — se renderizan como imágenes estáticas.

### Opción C: Exportar solo la tabla de actores

```python
# La tabla CSV ya fue guardada en la celda anterior
# Está en: results/04_final_actor_table.csv
```

Esta tabla es la que se puede importar directamente a herramientas de análisis externas o incluir en reportes de Word/PowerPoint.

In [ ]:
print("Archivos listos para exportar:")
print(f"  {RESULTS_DIR / '04_final_actor_table.csv'}")
print(f"  {RESULTS_DIR / '04_final_actor_table.parquet'}")
print()
print("Para exportar a HTML:")
print("  jupyter nbconvert --to html 04_sintesis_informe.ipynb")
print()
print("Para exportar a PDF:")
print("  jupyter nbconvert --to pdf 04_sintesis_informe.ipynb")
print()
print("=" * 50)
print("FIN DEL CASO CARDINGFORUMS")
print("=" * 50)